# Neural Machine Translation - Fine-Tuning with LLaMA-Factory

**Model**: Mistral-7B-Instruct-v0.3  
**Method**: QLoRA (4-bit, rank=64)  
**Data**: 3.7M translation pairs

## IMPORTANT: Setup Instructions
1. Go to `Runtime > Change runtime type > T4 GPU` (or A100 if Pro+)
2. Upload `train.jsonl` to Google Drive BEFORE running
3. Run cells in order
4. Training takes ~40-60 hours on T4, ~8-10 hours on A100

## 1. Install Dependencies

In [ ]:
# Install LLaMA-Factory and dependencies
!pip install -q transformers accelerate bitsandbytes peft trl datasets
!pip install -q sacrebleu sentencepiece tqdm

# Clone LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,metrics]" -q

# Verify GPU
!nvidia-smi

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else 'N/A')

## 2. Mount Google Drive & Setup Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project directory
!mkdir -p /content/nmt-project/data
!mkdir -p /content/nmt-project/outputs

In [ ]:
# Copy training data from Google Drive
# IMPORTANT: Upload train.jsonl to your Google Drive first!

import os

# Option 1: If you uploaded to root of Drive
DRIVE_PATH = "/content/drive/MyDrive/train.jsonl"

# Option 2: If you uploaded to a folder
# DRIVE_PATH = "/content/drive/MyDrive/NMT/train.jsonl"

if os.path.exists(DRIVE_PATH):
    !cp "{DRIVE_PATH}" /content/nmt-project/data/train.jsonl
    print("✓ Training data copied!")
else:
    print(f"ERROR: File not found at {DRIVE_PATH}")
    print("Please upload train.jsonl to Google Drive and update DRIVE_PATH")

# Verify
!wc -l /content/nmt-project/data/train.jsonl
!head -n 1 /content/nmt-project/data/train.jsonl

## 3. Create Dataset Config

In [ ]:
import json

# Create dataset_info.json for LLaMA-Factory
dataset_info = {
    "nmt_train": {
        "file_name": "/content/nmt-project/data/train.jsonl",
        "formatting": "alpaca",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output"
        }
    }
}

with open("/content/LLaMA-Factory/data/dataset_info.json", "r") as f:
    existing = json.load(f)

existing.update(dataset_info)

with open("/content/LLaMA-Factory/data/dataset_info.json", "w") as f:
    json.dump(existing, f, indent=2)

print("✓ Dataset config added!")

## 4. Login to Hugging Face

Required to download Mistral model. Get token from: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
login()  # Enter your HF token when prompted

## 5. Create Training Config

In [ ]:
training_config = """
### Model
model_name_or_path: mistralai/Mistral-7B-Instruct-v0.3

### Method (QLoRA)
stage: sft
do_train: true
finetuning_type: lora
lora_target: all
lora_rank: 64
lora_alpha: 128
lora_dropout: 0.05

### Quantization (4-bit)
quantization_bit: 4
quantization_method: bitsandbytes

### Dataset
dataset: nmt_train
template: mistral
cutoff_len: 512
max_samples: 100000
overwrite_cache: true
preprocessing_num_workers: 4

### Output (saves to Google Drive for persistence)
output_dir: /content/drive/MyDrive/nmt-outputs/mistral-nmt-qlora
logging_steps: 50
save_steps: 500
save_total_limit: 3
plot_loss: true
overwrite_output_dir: true

### Training hyperparameters
per_device_train_batch_size: 2
gradient_accumulation_steps: 16
learning_rate: 2.0e-4
num_train_epochs: 3
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true

### Optimization
optim: paged_adamw_32bit
max_grad_norm: 1.0

### Checkpointing (critical for Colab!)
save_safetensors: true
resume_from_checkpoint: true

### Evaluation
val_size: 0.01
per_device_eval_batch_size: 2
eval_strategy: steps
eval_steps: 500
"""

with open("/content/train_config.yaml", "w") as f:
    f.write(training_config)

print("✓ Training config created!")
print("\nConfig:")
print(training_config)

## 6. Start Training

⚠️ **IMPORTANT**: 
- Training will take many hours
- Checkpoints save to Google Drive every 500 steps
- If session disconnects, re-run cells 1-5, then this cell to resume

In [ ]:
# Start training!
%cd /content/LLaMA-Factory
!llamafactory-cli train /content/train_config.yaml

## 7. After Training: Export Model

In [ ]:
# Merge LoRA adapter with base model (optional)
export_config = """
model_name_or_path: mistralai/Mistral-7B-Instruct-v0.3
adapter_name_or_path: /content/drive/MyDrive/nmt-outputs/mistral-nmt-qlora
template: mistral
finetuning_type: lora
export_dir: /content/drive/MyDrive/nmt-outputs/mistral-nmt-merged
export_size: 2
export_device: cpu
export_legacy_format: false
"""

with open("/content/export_config.yaml", "w") as f:
    f.write(export_config)

!llamafactory-cli export /content/export_config.yaml

## 8. Test the Fine-Tuned Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Load base model + LoRA adapter
base_model = "mistralai/Mistral-7B-Instruct-v0.3"
adapter_path = "/content/drive/MyDrive/nmt-outputs/mistral-nmt-qlora"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(model, adapter_path)

print("✓ Model loaded!")

In [ ]:
def translate(text, src="En", tgt="Vi"):
    prompt = f"[INST] Translate {src} to {tgt}\n\n{text} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Test translations
test_sentences = [
    "Hello, how are you today?",
    "The weather is beautiful.",
    "I love learning new languages."
]

for sent in test_sentences:
    print(f"EN: {sent}")
    print(f"VI: {translate(sent)}")
    print()